In [ ]:
import pandas as pd
import re
import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import umap
from sklearn.datasets import load_digits


import hdbscan
import sklearn.cluster as cluster
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score


In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")


df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")
df_col

In [ ]:
df0[["q45_clé", "q24_research_fields"]].loc[df0.q24_research_fields.str.contains("LS6 Immunité, infection et immunothérapie")]

# Profil disciplinaire

In [ ]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column]= df0.apply(lambda row: row[column].replace(";","|") ,1 )
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data, df_explode

In [ ]:
df0["q24_research_fields"]= df0.apply(lambda row: row.q24_research_fields.replace(";","|") ,1 )


In [ ]:
gb_data, df_exp = split_multiple_choices(df0, column='q24_research_fields', index = "q45_clé")


In [ ]:
df_rf = df_exp[["q45_clé", 'q24_research_fields']].copy()
df_rf.loc[df_rf.q24_research_fields.str.contains("SH"), "q24_research_domain"] = "Sciences sociales et humaines"
df_rf.loc[df_rf.q24_research_fields.str.contains("LS"), "q24_research_domain"] = "Sciences de la vie"
df_rf.loc[df_rf.q24_research_fields.str.contains("PE"), "q24_research_domain"] = "Sciences physiques et ingénierie"
df_rf["value"] = 1
df_rf.q24_research_fields.value_counts()
df_rf

# Clusterisation des individus

Après le regroupement des champs disciplinaires, nous allons utilisés les mêmes algorithmes pour, cette fois, identifier les individus aux profils disciplinaires proches. L'exploration des données a mis en évidence l'existence de chercheurs qui "cochent" toutes les disciplines ou qui ont répondu deux fois (sans sélectionner le même nombre de disciplines). Nous commençons par enlevés ces lignes qui créent du bruit.

Ensuite, nous avons commencé par une approche "non-supervisée", puis au fur et à mesure de la classification des individus, nous avons utilisés les labels pour contruire des modèles semi-supervisé. Au final, nous avons identifiés neuf aires disciplinaires. Ces informations sont contenues dans les fichiers "tableau_cluster_ind.txt" et "tableau_cluster.txt". Le premier groupe que nous avons identifié



### Unsupervised classification

In [ ]:
df_rfi = df_rf.pivot(index =['q45_clé'], columns= 'q24_research_fields', values = "value").fillna(0).reset_index()
m_row = df_rfi[df_rfi.columns[1:]].values



In [ ]:
fig, ax = plt.subplots(1, figsize=(10,10))



embedding = umap.UMAP(n_neighbors=5,
                      min_dist=0.1,
                      n_components=2,
                      metric='cosine', random_state =42).fit_transform(m_row)

sns.scatterplot(x=embedding[:,0], y=embedding[:,1],  ax=ax)


In [ ]:
labels = hdbscan.HDBSCAN(
    min_samples=4,
    min_cluster_size=4, gen_min_span_tree=True
).fit(embedding)

labels.single_linkage_tree_.plot()
print(len(set(labels.labels_)))
print(len([x for x in labels.labels_ if x == -1]))

HDBSCAn détecte 13 clusters (en comptant les outliers). Il y a 7 outliers. Nous allons examiner les résultats en les enregistant dans un fichier texte, plus facile à annoter. Pour nous aider à labelliser les points, nous allons ajouter des infor sur le labo, le nom et le prénom (permet de rechercher les info sur Internet).

Attention : les données ne sont donc plus pseudonymisées (faire cette partie en local)

In [ ]:
dfn = pd.read_csv("buparis8_chercheurs_besoins_accompagnement_5-11-2026_16_7.csv", sep =";")
dfn = dfn.rename(columns={"142. Nom :":"q42_nom", "143. Prénom :":"q42_prenom", "146. Clé":"q45_clé"})

affil = pd.read_csv("list_affiliation.csv", sep =",")
affil1 = affil.merge(dfn[["q45_clé","q42_nom","q42_prenom"]], on = ["q45_clé"], how = "left")

In [ ]:
def write_cluster1(data, dict_clusters, data_affil=None, filename = "tableau_codage_cluster1.md"): 
    with open(filename, 'w') as fout:
        if "-1" in labels.labels_:
            range_cluster = [-1,len(set(labels.labels_))-1]
        else:
            range_cluster = [0,len(set(labels.labels_))]
        for x in range(range_cluster[0],range_cluster[1]):
            fout.write(f"Cluster {x} :\n\n")
            name_column = data.columns[1:22]
            abrev = "-|".join([re.search(r"\w*\d+", x).group() for x in name_column ])
            table_frame = "|".join(["----" for col in name_column])
            fout.write(f"|{abrev}|key|nom|ufr_labo|cluster|\n")
            fout.write(f"|{table_frame}|----|----|---|---|\n")
    
            compteur = 0
            list_row = []
            for v in dict_clusters:
                if dict_clusters[v] == x:
                    compteur+=1
                    if data_affil.empty:
                        dtmp1 = data[data.columns[1:22]].loc[data.q45_clé == v].values
                    #fout.write(f'{v}: {",".join([str(x) for x in dtmp1[0]])} | )
                        fout.write(f'|{"-|".join([str(x).replace("0.0","---") for x in dtmp1[0]])}|{v}| \n')
                    
                    else:
                        dtmp0 = data.merge(data_affil, on = "q45_clé", how = "left")
                        nom = dtmp0[dtmp0.columns[29:]].loc[dtmp0.q45_clé == v].values
                        ufr = dtmp0[dtmp0.columns[24]].loc[dtmp0.q45_clé == v].values
                        try:
                            old_cluster = dtmp0[dtmp0.columns[23]].loc[dtmp0.q45_clé == v].values
                        except:
                            old_cluster = ["No cluster"]
                        dtmp1 = dtmp0[dtmp0.columns[1:22]].loc[dtmp0.q45_clé == v].values
                        #fout.write(f'{v}: {",".join([str(x) for x in dtmp1[0]])} | )
                        fout.write(f'|{"-|".join([str(x).replace("0.0","---") for x in dtmp1[0]])}|{v}|{" ".join([str(x) for x in nom[0]])} | {ufr[0]}|{old_cluster[0]}|\n')
                        
                    list_row.append(dtmp1)
            try:
                total = list(np.sum(list_row, axis=0))
                fout.write(f'|{"-|".join([str(x) for x in total[0]])}|Total|---|---|---|\n')
            except:
                fout.write(f'\n')
            fout.write("\n=====================\n")



    

In [ ]:
dtmp0[dtmp0.columns[1:22]].loc[dtmp0.q45_clé == v].values

### Retrait des doublons et lignes atypiques

L'exploration des résultats de la clusterisation a permi de noter la présence de deux doublons : deux personnes ont répondu deux fois au questionnaire. Deux autres chercheurs ont un profil atypique. L'un n'a sélectionné qu'une discipline : "chimie de synthèse et des matériaux". Un autre a sélectionné 16 des 21 champ disciplinaires propopsés. Nous allons donc refaire tourner les algorithmes en enlevant ces quatre lignes de notre échantillon. 


In [ ]:
df_rfi1 = df_rfi.loc[~df_rfi.q45_clé.isin(["9EAP-NB4B","QNYZ-3MH2","H5X6-KL2Z",'PS8W-TJ9P'])] #

In [ ]:
fig, ax = plt.subplots(1, figsize=(10,10))



m_row1= df_rfi1[df_rfi1.columns[1:]].values

embedding = umap.UMAP(n_neighbors=4,
                      min_dist=0.1,
                      n_components=2,
                      metric='cosine', random_state =42).fit_transform(m_row1)




labels = hdbscan.HDBSCAN(
    min_samples=4,
    min_cluster_size=4, gen_min_span_tree=True
).fit(embedding)

#labels.single_linkage_tree_.plot()
print("Nombre de clusters :", len(set(labels.labels_))-1)
print("Nombre d'outliers :", len([x for x in labels.labels_ if x == -1]))

sns.scatterplot(x=embedding[:,0], y=embedding[:,1],  ax=ax, hue= [str(x) for x in labels.labels_])

In [ ]:
dict_clusters = {}

for n, x in enumerate(df_rfi1.q45_clé):
    dict_clusters[x] = labels.labels_[n]
    
write_cluster(df_rfi1, dict_clusters = dict_clusters, data_affil = affil1, filename="tableau_codage_cluster_total.md")

## Semisupervised dimension reduction

À partir des premières classifications non supervisées, nous allons affinés le modèle en prenant en compte les clusters identifiés et les groupes que nous avons repérés à la lecture des résultats de la clusterisation. À ce stade, le principe est de classer uniquement les points qui ne posent pas de difficultés.

In [ ]:
with open("tableau_codage_cluster_semi_v6.md", "r") as fin:
    lines = fin.readlines()

new_dict_cluster = {}
dict_nom_cluster = {}
for l in lines:
    if re.search("Cluster", l):
        nom_cluster = l.split(":")[-1].strip()
        no_cluster = int(re.search(r"-*\d+", l.split(":")[0]).group())
        dict_nom_cluster[no_cluster]= nom_cluster
    if "|" in l:
        split_l = l.strip().split("|")
        if len(split_l[-1]) > 0:
            id_row = split_l[22]
            new_dict_cluster[id_row] = int(split_l[-1].strip())

print(len(new_dict_cluster))
new_dict_cluster

dict_nom_cluster

In [ ]:
fig, ax = plt.subplots(1, figsize=(10,10))

df_rfi1["no_cluster"] = df_rfi1.q45_clé.map(new_dict_cluster.get)
df_rfi1["nom_cluster"] = df_rfi1.no_cluster.map(dict_nom_cluster.get)

m_row1= df_rfi1[df_rfi1.columns[1:22]].values
target_labels = df_rfi1[df_rfi1.columns[22]].values

embedding = umap.UMAP(n_neighbors=7,
                      min_dist=0.1,
                      n_components=3,
                      metric='cosine', random_state =42).fit_transform(m_row1, target = target_labels)




labels = hdbscan.HDBSCAN(
    min_samples=4,
    min_cluster_size=4, gen_min_span_tree=True
).fit(embedding)

#labels.single_linkage_tree_.plot()
print("Nombre de clusters :", len(set(labels.labels_))-1)
print("Nombre d'outliers :", len([x for x in labels.labels_ if x == -1]))

sns.scatterplot(x=embedding[:,0], y=embedding[:,1],  ax=ax, hue= [str(x) for x in labels.labels_])

La nouvelle clusteristation, on recommence l'étiquetage des points. S'il reste encore des point non labellisés, on réitère la clusterisation semi-supervisé jusqu'à atteindre une classification satisfaisante. La visualisation de la spatialisation dans un graphique interactif peut aider à analyser le résultat.

In [ ]:
import plotly.express as px

dict_clusters = {}

for n, x in enumerate(df_rfi1.q45_clé):
    dict_clusters[x] = str(labels.labels_[n])

dict_nom = dict(zip(affil1.q45_clé, affil1.q42_nom))
dict_prenom = dict(zip(affil1.q45_clé, affil1.q42_prenom))

list_row = []
for n, x in enumerate(embedding):
    id_key = df_rfi1[df_rfi1.columns[0]].values[n]
    try:
        nom = dict_nom[id_key].lower()
    except:
        nom = "not defined"
    try:
        prenom = dict_prenom[id_key].lower()
    except:
        prenom = "not defined"
    dict_row = {"q45_clé": id_key,
                      "x": x[0],
                      "y": x[1],
                      "nom": nom,
                      "prenom": prenom
                     }
    list_row.append(dict_row)
    

df_emb = pd.DataFrame.from_dict(list_row)
df_emb["no_cluster"] = df_emb.q45_clé.map(dict_clusters.get)




fig_2d = px.scatter(
    df_emb, x="x", y="y",
    color=df_emb.no_cluster, hover_data=['q45_clé', 'nom', 'prenom'],
)

fig_2d.write_html('plotly_embedding_codage.html')

In [ ]:
dict_clusters = {}

for n, x in enumerate(df_rfi1.q45_clé):
    dict_clusters[x] = labels.labels_[n]
    
write_cluster1(df_rfi1, dict_clusters = dict_clusters, data_affil = affil1, filename="tableau_codage_cluster_semi_v8.md")

### Projection des points avec les labels

In [ ]:
with open("../data/clusterisation/7_tableau_codage_cluster_v7.md", "r") as fin:
    lines = fin.readlines()

new_dict_cluster = {}
dict_nom_cluster = {}
for l in lines:
    if re.search("Cluster", l):
        nom_cluster = l.split(":")[-1].strip()
        no_cluster = int(re.search(r"-*\d+", l.split(":")[0]).group())
        dict_nom_cluster[no_cluster]= nom_cluster
    if "|" in l:
        split_l = l.strip().split("|")
        if len(split_l[-1]) > 0:
            id_row = split_l[22]
            new_dict_cluster[id_row] = int(split_l[-1].strip())

print(len(new_dict_cluster))
new_dict_cluster

dict_nom_cluster

In [ ]:
fig, ax = plt.subplots(1, figsize=(10,10))

df_rfi1["no_cluster"] = df_rfi1.q45_clé.map(new_dict_cluster.get)
df_rfi1["nom_cluster"] = df_rfi1.no_cluster.map(dict_nom_cluster.get)

m_row1= df_rfi1[df_rfi1.columns[1:22]].values
target_labels = df_rfi1[df_rfi1.columns[22]].values

embedding = umap.UMAP(n_neighbors=7,
                      min_dist=0.15,
                      n_components=2,
                      metric='cosine', random_state =42).fit_transform(m_row1, target = target_labels)




sns.scatterplot(x=embedding[:,0], y=embedding[:,1],  ax=ax, hue= [str(x) for x in target_labels])

In [ ]:
import plotly.express as px

dict_clusters = {}

for n, x in enumerate(df_rfi1.q45_clé):
    dict_clusters[x] = str(labels.labels_[n])

dict_nom = dict(zip(affil1.q45_clé, affil1.q42_nom))
dict_prenom = dict(zip(affil1.q45_clé, affil1.q42_prenom))

list_row = []
for n, x in enumerate(embedding):
    id_key = df_rfi1[df_rfi1.columns[0]].values[n]
    dict_row = {"q45_clé": id_key,
                      "x": x[0],
                      "y": x[1]
                     }
    list_row.append(dict_row)
    

df_emb = pd.DataFrame.from_dict(list_row)
df_emb["no_cluster"] = df_emb.q45_clé.map(new_dict_cluster.get)
df_emb["nom_cluster"] = df_emb.no_cluster.map(dict_nom_cluster.get)



fig_2d = px.scatter(
    df_emb, x="x", y="y",
    color=df_emb.nom_cluster, hover_data=['q45_clé'],
)

fig_2d.write_html('plotly_embedding_codage.html')

# Enregistrement et Pseudonymisation

Le processus itératif de classification aparraît à travers les différents fichiers dont le nom commence par "pseudo_0_tableau_codage_cluster" (v0, v1, etc indiquent l'ordre du codage).En tout, nous avons procédé à 9 classifications différentes. Sauf pour les classification 0 et 1 qui ne sont pas semisupervisées, les autres sont établies à partir de la classification directement antérieur. La classification v2 s'appuie sur v1, v3 sur v2, etc.

À partir de la classification v4, chaque itération correspond à une "classification hiérarchique ascendante" dans la mesure où on procède également à des regroupements. On passe de 18 classe à 14, puis 12, puis, 10, puis 8. Par exemple, la dernière classification (v7) est la plus englobantes : certains clusters apparaissant dans la classification v6 ont été regroupée dans des ensembles plus larges: le cluster "Art, cultures et communication" comprends des chercheurs classés auparavant dans "humanité", "arts et sociétés", etc.

|No classification|nombre de clusters|
|:---:|:---:|
|classif_0|12|
|classif_1|12|
|classif_2|15|
|classif_3|18|
|classif_4|18|
|classif_5|14|
|classif_6|12|
|classif_7|10|
|classif_8|8|

L'historique de la classification est également enregistré de façon plus synthétique dans le fichier "research_field_ind_VF.csv". Il contient les coordonnées des points, ainsi que le nom des différents clusters dans lesquels ils ont été classés.



In [ ]:
import os

os.getcwd()

In [ ]:
list_col = ['LS4', 'LS5', 'LS6', 'LS7', 'LS8', 'LS9', 'PE1', 'PE10', 'PE5', 'PE6', 'PE7', 'PE8', 'PE9', 'SH1', 'SH2', 'SH3', 'SH4', 'SH5', 'SH6', 'SH7', 'SH8', 'key', 'nom', 'ufr_labo', 'cluster']
list_file = [x for x in os.listdir("../data/clusterisation/") if ".md" in x ]

dict_row ={}
for compteur in range(8):
    print(compteur)
    for f in list_file:
        if re.match(str(compteur), f):
            with open("../data/clusterisation/"+f, 'r') as fin :
                lines = fin.readlines()
            for l in lines :
                if "|" in l:
                    if re.search("cluster",l):
                        pass
                    elif re.search("key",l):
                        pass
                    elif  not re.search(r"\d",l) :
                        pass
                    elif re.search("Total", l):
                        pass
                    else:
                        #print(l)
                        split_l =  [x for x in l.strip().split("|") if len(x) > 0]
                        new_l = [x for x in split_l if split_l.index(x) !=23]
                        id_row = split_l[21]
                        if compteur == 0:
                            fields = [n for n, x in enumerate(split_l) if re.search("1.0-", x)]
                            #id_fields = [split_l.index(x) for x in split_l if re.search("1.0-", x)]
                            selected_field = [list_col[x] for x in fields]
                            #print(len(split_l),id_row, fields, selected_field)
                            dict_row[id_row] = [selected_field]
                            
                        else:
                            #print(id_row, dict_row[id_row])
                            list_row = dict_row[id_row]
                            #print(list_row)
                            nom_cluster = split_l[24]
                            
                            if nom_cluster == "None":
                                nom_cluster = "Outliers"
                            else:
                                pass
                            #print(list_row)
                            list_row.append(nom_cluster)
                            dict_row[id_row] = list_row
    
                            
row_df = []

for x in dict_row:
    row= {"q45_clé": x,
        "fields" : dict_row[x][0],
          "classif_0": dict_row[x][1],
          "classif_1": dict_row[x][2],
          "classif_2": dict_row[x][3],
          "classif_3": dict_row[x][4],
          "classif_4": dict_row[x][4],
          "classif_5": dict_row[x][5],
          "classif_6": dict_row[x][6],
         "classif_7": dict_row[x][7]}
    row_df.append(row)
          

In [ ]:
clusterisation = pd.DataFrame.from_dict(row_df)
clusterisation["classif_8"] = clusterisation.q45_clé.map(dict(zip(df_emb.q45_clé, df_emb.nom_cluster)).get)



In [ ]:
df_emb1= df_emb.merge(clusterisation, on = ["q45_clé"], how = "left")
df_emb1.to_csv("../data/clusterisation/research_field_ind_VF.csv", sep =",", index = False)

fig_2d = px.scatter(
    df_emb1, x="x", y="y",
    color=df_emb1.classif_6, hover_data=['q45_clé'],
)

fig_2d.write_html('plotly_embedding_codage.html')

In [ ]:
list_col = ['LS4', 'LS5', 'LS6', 'LS7', 'LS8', 'LS9', 'PE1', 'PE10', 'PE5', 'PE6', 'PE7', 'PE8', 'PE9', 'SH1', 'SH2', 'SH3', 'SH4', 'SH5', 'SH6', 'SH7', 'SH8', 'key', 'nom', 'ufr_labo', 'cluster']
list_file = [x for x in os.listdir("../data/clusterisation/") if ".md" in x and "pseudo" not in x ]

list_new_line = []
for compteur in range(8):
    print(compteur)
    for f in list_file:
        with open("../data/clusterisation/"+f, 'r') as fin :
            lines = fin.readlines()
        with open("../data/clusterisation/pseudo_"+f, 'w', newline='') as fout:
            for l in lines :
                
                if "|" in l:
                    print(l)
                    split_l =  [x for x in l.strip().split("|") if len(x) > 0]
                    new_l = [x for x in split_l if split_l.index(x) !=22 and  split_l.index(x) !=23]
                    #print("|".join(new_l))
                    fout.write("|"+"|".join(new_l)+"|\n")
                else:
                   #print(l)
                    fout.write(l)
                        
    
                 
